# 🎾 Bloque 2 — Exploración de datos (EDA) con DuckDB

**Objetivo:** entender los datos *antes* de construir el pipeline. Cada "trampa" que encontremos
aquí se convierte en una regla de limpieza (silver), un test (dbt) o una precaución contra el
*data leakage* (feature store).

**Fuentes que comparamos**

| Fuente | Qué es |
|---|---|
| **Archivo Sackmann** (`Aneeshers/tennis-sackmann-archive`) | Copia congelada del dataset de referencia de Jeff Sackmann: partidos ATP desde 1968, jugadores, rankings |
| **TML-Database** | Proyecto que continuaba el esquema de Sackmann con actualizaciones diarias |
| **tennis-data.co.uk** | Resultados + cuotas de apuestas desde 2000 (Excel) |

**Cómo leer este notebook:** cada sección sigue el patrón *pregunta → SQL → resultado → conclusión*.
Ejecuta las celdas en orden (`Shift+Enter`).

> Datos: Jeff Sackmann / TML-Database (CC BY-NC-SA 4.0) y tennis-data.co.uk. No se guardan en el repo.

## 0. Preparación

### ¿Qué es DuckDB y por qué aquí?
DuckDB es una base de datos SQL **analítica** (OLAP, almacenamiento por columnas) que corre
**dentro de Python**, sin servidor. Consulta ficheros CSV, Parquet o Excel directamente con
`FROM 'fichero.csv'`. Para explorar es ideal: rapidísimo, gratis y sin gastar cuota de Databricks.

La siguiente celda descarga los datos a `data/sample/` (ignorada por git). Tiene **caché**: si el
fichero ya existe no lo vuelve a descargar — un primer contacto con la idea de *idempotencia*
(ejecutar dos veces da el mismo resultado sin trabajo duplicado).

In [1]:
import urllib.request
from pathlib import Path

import duckdb
import pandas as pd

# The notebook may run from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
DATA = ROOT / "data" / "sample"

ARCHIVE = "https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/main/atp"
TML = "https://raw.githubusercontent.com/Tennismylife/TML-Database/master"
ODDS = "https://www.tennis-data.co.uk/hrjk-85HytOjkhth76j_ygh4jf7"
# tennis-data.co.uk answers 403 to Python's default user-agent (found in Block 0)
HEADERS = {"User-Agent": "Mozilla/5.0"}

files = {
    f"sackmann/atp_matches_{y}.csv": f"{ARCHIVE}/atp_matches_{y}.csv" for y in range(1968, 2027)
}
for name in [
    "atp_matches_qual_chall_2025.csv",
    "atp_matches_qual_chall_2026.csv",
    "atp_players.csv",
    "atp_rankings_current.csv",
    "atp_rankings_20s.csv",
]:
    files[f"sackmann/{name}"] = f"{ARCHIVE}/{name}"
for name in ["2025.csv", "2026.csv", "ATP_Database.csv"]:
    files[f"tml/{name}"] = f"{TML}/{name}"
for y in [2025, 2026]:
    files[f"odds/{y}.xlsx"] = f"{ODDS}/{y}/{y}.xlsx"

for rel_path, url in files.items():
    target = DATA / rel_path
    if target.exists():
        continue  # cache hit: never download twice
    target.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=60) as resp:
        target.write_bytes(resp.read())

size_mb = sum(f.stat().st_size for f in DATA.rglob("*") if f.is_file()) / 1e6
print(f"{len(files)} files ready in data/sample ({size_mb:.1f} MB)")

69 files ready in data/sample (57.8 MB)


In [2]:
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 60)

con = duckdb.connect()  # in-memory database: nothing is written to disk
D = DATA.as_posix()  # forward slashes, so paths work inside SQL on Windows


def sql(query: str) -> pd.DataFrame:
    """Run a query in DuckDB and return the result as a pandas DataFrame."""
    return con.sql(query).df()

### Vistas: "tablas" que leen los ficheros en cada consulta
Una **vista** (`CREATE VIEW`) es una consulta guardada con nombre. No copia datos: cada vez que
consultas `matches`, DuckDB lee los CSV. Fíjate en el **glob** `atp_matches_[0-9][0-9][0-9][0-9].csv`:
lee los 59 ficheros de años de una vez (y excluye `qual_chall`).

- `union_by_name=true` → une ficheros por **nombre** de columna, no por posición (si un año tuviera
  columnas en otro orden, no se mezclarían).
- `filename=true` → añade una columna con el fichero de origen: **linaje** a nivel de fila.

In [3]:
con.sql(f"""
CREATE OR REPLACE VIEW matches AS
SELECT * FROM read_csv('{D}/sackmann/atp_matches_[0-9][0-9][0-9][0-9].csv',
                       union_by_name = true, filename = true)
""")
con.sql(f"CREATE OR REPLACE VIEW players AS SELECT * FROM read_csv('{D}/sackmann/atp_players.csv')")
con.sql(f"""CREATE OR REPLACE VIEW qual_chall AS
    SELECT * FROM read_csv('{D}/sackmann/atp_matches_qual_chall_*.csv', union_by_name = true)""")
con.sql(
    f"CREATE OR REPLACE VIEW rankings_20s AS SELECT * FROM read_csv('{D}/sackmann/atp_rankings_20s.csv')"
)
con.sql(
    f"CREATE OR REPLACE VIEW rankings_current AS SELECT * FROM read_csv('{D}/sackmann/atp_rankings_current.csv')"
)
print("views created")

views created


## 1. Primer vistazo

**Pregunta:** ¿qué columnas hay, de qué tipo, y cuántos partidos por época?

`DESCRIBE` muestra el esquema que DuckDB ha **inferido** leyendo una muestra del CSV (un CSV no
guarda tipos: todo es texto; el lector adivina). Ojo: `tourney_date` se infiere como número
(`20250526`), no como fecha — la convertiremos en silver.

In [4]:
sql("SELECT column_name, column_type FROM (DESCRIBE matches)").T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49
column_name,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,filename
column_type,VARCHAR,VARCHAR,VARCHAR,BIGINT,VARCHAR,BIGINT,BIGINT,BIGINT,BIGINT,VARCHAR,VARCHAR,VARCHAR,BIGINT,VARCHAR,DOUBLE,BIGINT,BIGINT,VARCHAR,VARCHAR,VARCHAR,BIGINT,VARCHAR,DOUBLE,VARCHAR,BIGINT,VARCHAR,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,BIGINT,VARCHAR


`SUMMARIZE` es el **perfilado automático** de DuckDB: por columna da mínimo, máximo, nº de valores
distintos aproximado, media y **% de nulos**. Es lo primero que se hace con un dataset nuevo.

In [5]:
sql("""
SELECT column_name, column_type, min, max, approx_unique, null_percentage
FROM (SUMMARIZE matches)
WHERE column_name IN ('tourney_date', 'surface', 'tourney_level', 'round', 'best_of', 'score',
                      'minutes', 'w_ace', 'winner_rank', 'winner_ht', 'winner_age')
""")

,column_name,column_type,min,max,approx_unique,null_percentage
0,surface,VARCHAR,Carpet,Hard,3,1.50
1,tourney_level,VARCHAR,A,O,6,0.00
2,tourney_date,BIGINT,19671228,20260525,3728,0.00
3,winner_ht,BIGINT,3,211,38,8.40
4,winner_age,DOUBLE,14.3,58.7,339,0.66
5,score,VARCHAR,4-3 6-6,Walkover,22932,0.00
6,best_of,BIGINT,1,5,3,0.00
7,round,VARCHAR,BR,SF,10,0.00
8,minutes,BIGINT,0,1146,351,50.37
9,w_ace,BIGINT,0,113,54,48.88


In [6]:
sql("""
SELECT (tourney_date // 100000) * 10 AS decade,
       count(*)                      AS n_matches,
       min(tourney_date)             AS first_date,
       max(tourney_date)             AS last_date
FROM matches
GROUP BY ALL
ORDER BY decade
""")

,decade,n_matches,first_date,last_date
0,1960,7542,19671228,19691226
1,1970,39181,19700104,19791231
2,1980,36217,19800101,19891215
3,1990,37150,19900101,19991203
4,2000,32335,20000103,20091204
5,2010,29397,20100103,20191124
6,2020,17567,20200106,20260525


**Conclusión:** ~199.000 partidos del circuito principal, del **28-dic-1967 al 25-may-2026**
(Roland Garros 2026). Los años 70 tienen más partidos que hoy: había muchos más torneos pequeños.

> 💡 **Lección de SQL en DuckDB:** escribe siempre `AS` en los alias. Sin `AS`, palabras como
> `weeks`, `years` o `start` se interpretan como unidades de tiempo y la consulta falla
> (nos pasó explorando).

## 2. Grano y clave primaria

El **grano** de una tabla es "qué representa una fila". Aquí: **un partido**, siempre escrito
desde el punto de vista del ganador (`winner_*`) y el perdedor (`loser_*`).

**Pregunta:** ¿`tourney_id + match_num` identifica un partido de forma única? Si hubiera
duplicados, los `JOIN` posteriores multiplicarían filas sin avisar.

In [7]:
sql("""
SELECT count(*)                                     AS n_rows,
       count(DISTINCT (tourney_id, match_num))      AS n_distinct_keys,
       count(*) - count(DISTINCT (tourney_id, match_num)) AS n_duplicates
FROM matches
""")

,n_rows,n_distinct_keys,n_duplicates
0,199389,199389,0


**Conclusión:** ✅ clave única, **0 duplicados**. En dbt lo fijaremos con un test
`unique_combination_of_columns` para que, si algún día entra un duplicado, el pipeline falle.

⚠️ **Trampa de grano:** el ganador siempre va en las columnas `winner_*`. Si entrenáramos un modelo
con "jugador 1 = ganador", la etiqueta sería siempre 1 → *leakage* total. En el Bloque 11
**simetrizaremos** (jugador A/B asignado al azar).

## 3. La trampa de las fechas 📅

**Pregunta:** ¿`tourney_date` es el día en que se jugó el partido?

In [8]:
sql("""
SELECT round,
       min(tourney_date) AS min_date,
       max(tourney_date) AS max_date,
       count(*)          AS n_matches,
       min(match_num)    AS min_match_num,
       max(match_num)    AS max_match_num
FROM matches
WHERE tourney_name = 'Roland Garros' AND tourney_date // 10000 = 2025
GROUP BY ALL
ORDER BY min_match_num
""")

,round,min_date,max_date,n_matches,min_match_num,max_match_num
0,R128,20250526,20250526,64,275,338
1,R64,20250526,20250526,32,339,370
2,R32,20250526,20250526,16,371,386
3,R16,20250526,20250526,8,387,394
4,QF,20250526,20250526,4,395,398
5,SF,20250526,20250526,2,399,400
6,F,20250526,20250526,1,401,401


**Conclusión:** ❌ **No.** Las 127 rondas de Roland Garros 2025, de la primera ronda a la final,
tienen la misma fecha: `20250526`, el **lunes de inicio del torneo**. La final se jugó el 8 de junio.

**Por qué importa (data leakage):** si calculamos "forma del jugador antes del partido" filtrando
por `tourney_date < fecha`, en la final estaríamos ignorando los 6 partidos anteriores del
torneo... y si filtramos por `<=`, en primera ronda estaríamos usando resultados que aún no
habían ocurrido. **Solución (Bloque 6):**
- `round_order`: R128=1 → F=7.
- `match_seq`: orden cronológico total = (`tourney_date`, `tourney_id`, `round_order`, `match_num`).
- `est_match_date`: fecha **estimada** según la ronda (para la fatiga: "minutos jugados en 7 días").

**¿Basta `match_num` para ordenar?** En RG 2025 crece con la ronda (la final es el 401). Pero:

In [9]:
sql("""
WITH r AS (
    SELECT tourney_id, round,
           CASE round WHEN 'R128' THEN 1 WHEN 'R64' THEN 2 WHEN 'R32' THEN 3 WHEN 'R16' THEN 4
                      WHEN 'QF' THEN 5 WHEN 'SF' THEN 6 WHEN 'F' THEN 7 END AS round_order,
           min(match_num) AS min_num, max(match_num) AS max_num
    FROM matches
    WHERE round IN ('R128', 'R64', 'R32', 'R16', 'QF', 'SF', 'F')
    GROUP BY ALL
)
SELECT count(DISTINCT a.tourney_id) AS tourneys_where_match_num_is_not_chronological,
       (SELECT count(DISTINCT tourney_id) FROM r) AS total_tourneys
FROM r AS a
JOIN r AS b ON a.tourney_id = b.tourney_id AND b.round_order = a.round_order + 1
WHERE b.min_num < a.max_num
""")

,tourneys_where_match_num_is_not_chronological,total_tourneys
0,82,4621


**Conclusión:** en **82 de 4.621** torneos `match_num` **no** respeta el orden de rondas. Por eso
ordenamos **primero por ronda** y solo después por `match_num`.

## 4. Dominios de las columnas categóricas

**Pregunta:** ¿qué valores toma cada categoría? Esto define los tests `accepted_values` de dbt y
las tablas de referencia (*seeds*).

In [10]:
sql("""
SELECT 'surface' AS col, surface AS value, count(*) AS n FROM matches GROUP BY ALL
UNION ALL SELECT 'tourney_level', tourney_level, count(*) FROM matches GROUP BY ALL
UNION ALL SELECT 'best_of', best_of::VARCHAR, count(*) FROM matches GROUP BY ALL
UNION ALL SELECT 'winner_hand', winner_hand, count(*) FROM matches GROUP BY ALL
ORDER BY col, n DESC
""")

,col,value,n
0,best_of,3,156016
1,best_of,5,43337
2,best_of,1,36
3,surface,Hard,80877
4,surface,Clay,70920
5,surface,Grass,23702
6,surface,Carpet,20900
7,surface,NaN,2990
8,tourney_level,A,129719
9,tourney_level,G,28299


In [11]:
sql("""
SELECT round, count(*) AS n FROM matches GROUP BY ALL ORDER BY n DESC
""").T

,0,1,2,3,4,5,6,7,8,9
round,R32,R16,R64,RR,QF,R128,SF,F,BR,ER
n,64438,34702,33987,17824,17786,16776,9132,4649,63,32


In [12]:
sql("""
SELECT winner_entry, count(*) AS n FROM matches GROUP BY ALL ORDER BY n DESC
""").T

,0,1,2,3,4,5,6,7,8,9,10,11
winner_entry,NaN,Q,WC,LL,PR,SE,Alt,ALT,NG,W,ITF,UP
n,181484,11016,5244,1264,261,90,20,3,3,2,1,1


In [13]:
sql("""
SELECT tourney_level, count(*) AS n_matches_without_surface
FROM matches WHERE surface IS NULL GROUP BY ALL
""")

,tourney_level,n_matches_without_surface
0,A,1841
1,D,1149


**Conclusiones:**
- `tourney_level`: **G** Grand Slam · **M** Masters 1000 · **A** resto ATP · **F** Finals ·
  **D** Copa Davis · **O** Juegos Olímpicos.
- `surface`: Hard, Clay, Grass, **Carpet** (ya no se usa) y **2.990 nulos** (Copa Davis y algunos
  torneos antiguos) → para el Elo por superficie esos partidos solo cuentan en el Elo general.
- `round`: además de R128…F aparecen **RR** (round robin: Finals, United Cup), **BR** (bronce
  olímpico) y **ER** (rondas previas de Davis).
- `best_of = 1`: 36 partidos anómalos.
- `winner_entry`: valores **sucios** — `Alt` y `ALT` son lo mismo → normalizar mayúsculas en silver.
- `hand`: `U` = desconocida, `A` = ambidiestro.

## 5. Formatos de marcador (`score`)

**Pregunta:** ¿qué formatos aparecen? Necesitamos saberlo para escribir el *parser* del Bloque 6
(sets, juegos, tie-breaks, retiradas). Clasificamos con **expresiones regulares**:
- `regexp_full_match(score, '(\d+-\d+(\(\d+\))?\s*)+')` → "uno o más sets tipo `6-4` o `7-6(5)`".

In [14]:
SCORE_KIND = r"""
CASE
    WHEN score IS NULL OR trim(score) = ''                 THEN 'empty'
    WHEN score ILIKE '%W/O%'                               THEN 'walkover'
    WHEN score ILIKE '%RET%'                               THEN 'retirement'
    WHEN score ILIKE '%DEF%'                               THEN 'default'
    WHEN score ILIKE '%abandoned%' OR score ILIKE '%ABD%'  THEN 'abandoned'
    WHEN regexp_matches(score, '\[\d+-\d+\]')              THEN 'match_tiebreak'
    WHEN regexp_full_match(score, '(\d+-\d+(\(\d+\))?\s*)+')
        THEN CASE WHEN score LIKE '%(%' THEN 'normal_with_tiebreak' ELSE 'normal' END
    ELSE 'other'
END"""

sql(f"""
SELECT {SCORE_KIND} AS score_kind,
       count(*) AS n,
       round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct,
       any_value(score) AS example
FROM matches
GROUP BY ALL
ORDER BY n DESC
""")

,score_kind,n,pct,example
0,normal,153188,76.83,6-2 6-2
1,normal_with_tiebreak,40491,20.31,7-6(7) 2-6 6-4 6-1
2,retirement,4012,2.01,6-1 3-6 3-0 RET
3,walkover,1301,0.65,W/O
4,default,164,0.08,5-7 6-4 2-0 DEF
5,other,134,0.07,6-1 5-4 Played and unfinished
6,match_tiebreak,67,0.03,7-6(5) 5-7 [3-10]
7,abandoned,29,0.01,2-6 4-6 6-4 8-6 5-3 ABD
8,empty,3,0.00,NaN


In [15]:
sql(f"""
SELECT score, count(*) AS n
FROM matches
WHERE {SCORE_KIND} = 'other'
GROUP BY ALL
ORDER BY n DESC
LIMIT 12
""")

,score,n
0,UNK,96
1,NA,6
2,Walkover,6
3,6-6 Played and unfinished,2
4,5-6,2
5,7-6 4-6 6-1?,1
6,7-6 ?-?,1
7,4-3 6-6,1
8,6-3 6-7 4-6?,1
9,6-4 ?-?,1


**Conclusiones:**
- 97,1 % son marcadores normales (20 % con tie-break).
- **Retiradas** (`6-3 2-1 RET`, 4.012) y **walkovers** (`W/O`, 1.301): el partido no se completó.
  - Un **W/O no se jugó** → no debe contar para el Elo ni para la forma.
  - Una retirada sí se jugó parcialmente → decisión a documentar (ADR en el Bloque 8).
- **134 "other"**: `UNK` (marcador desconocido), `NA`, marcadores con `?` (dudosos) y
  "Played and abandoned". El parser debe tolerarlos y marcarlos (`score_parse_ok = false`),
  **nunca romper el pipeline**.

## 6. Estadísticas ausentes por época

**Pregunta:** ¿desde cuándo hay estadísticas de saque (aces, puntos de saque...) y ranking?
Condiciona qué features puede usar el modelo y desde qué año entrenar.

In [16]:
sql("""
SELECT (tourney_date // 100000) * 10               AS decade,
       round(100 * avg((w_ace IS NULL)::INT), 1)   AS pct_null_aces,
       round(100 * avg((minutes IS NULL)::INT), 1) AS pct_null_minutes,
       round(100 * avg((winner_rank IS NULL)::INT), 1) AS pct_null_rank,
       round(100 * avg((winner_ht IS NULL)::INT), 1)   AS pct_null_height
FROM matches
GROUP BY ALL
ORDER BY decade
""")

,decade,pct_null_aces,pct_null_minutes,pct_null_rank,pct_null_height
0,1960,100.0,100.0,100.0,40.1
1,1970,100.0,100.0,55.3,24.9
2,1980,100.0,100.0,14.5,4.1
3,1990,20.2,23.6,1.9,2.5
4,2000,11.7,11.7,1.0,2.9
5,2010,7.5,11.9,0.7,1.5
6,2020,6.1,8.0,0.5,1.0


**Conclusiones:**
- Estadísticas de saque: **inexistentes antes de 1991** (100 % nulos en los 80), ~80 % disponibles
  en los 90 y >90 % desde 2000.
- Ranking: el ranking ATP nació en **1973** → nulos altos en los 60–70.
- **Decisión:** el **Elo** se calcula desde 1968 (solo necesita quién ganó), pero el **modelo ML**
  se entrenará con partidos desde ~2000, donde las features de saque existen.

## 7. Jugadores

**Pregunta:** ¿todos los jugadores de los partidos existen en la tabla de jugadores? (integridad
referencial — en dbt será un test `relationships`). ¿Qué atributos faltan?

In [17]:
sql("""
SELECT count(*)                                   AS n_players,
       count(DISTINCT player_id)                  AS n_distinct_ids,
       round(100 * avg((dob IS NULL)::INT), 1)    AS pct_null_birthdate,
       round(100 * avg((height IS NULL)::INT), 1) AS pct_null_height,
       round(100 * avg((name_first IS NULL OR name_last IS NULL)::INT), 1) AS pct_null_name_part
FROM players
""")

,n_players,n_distinct_ids,pct_null_birthdate,pct_null_height,pct_null_name_part
0,66912,66912,27.9,93.8,1.4


In [18]:
sql("""
WITH match_players AS (
    SELECT winner_id AS player_id FROM matches
    UNION
    SELECT loser_id FROM matches
)
SELECT count(*) AS players_in_matches,
       count(p.player_id) AS found_in_players_table
FROM match_players AS m
LEFT JOIN players AS p USING (player_id)
""")

,players_in_matches,found_in_players_table
0,7709,7709


**Conclusiones:**
- ✅ **Integridad referencial perfecta**: todos los jugadores de los partidos existen en `players`.
- `height` nulo en el 94 % de la tabla (hay 67.000 jugadores, la mayoría de nivel bajo); en los
  partidos del circuito principal solo falta en ~1–3 %. Fecha de nacimiento nula en el 28 %.
- Algunos jugadores tienen nombre o apellido nulo → la clave de nombres para cruzar con las cuotas
  debe tolerarlo.

## 8. ⭐ Comparación de fuentes: ¿cuál usamos?

Esta es la sección clave del bloque. En el ADR-0002 elegimos TML porque "se actualizaba a diario".
**Verifiquémoslo con datos**, incluyendo cómo lee DuckDB los ficheros de TML.

### 8.1 Leer TML... y encontrar ficheros rotos
Usamos `store_rejects = true`: en lugar de fallar, DuckDB **aparta las filas inválidas** en la tabla
`reject_errors`. Es el patrón de **cuarentena** (*dead-letter*) que usan los pipelines reales:
no pierdes datos buenos por culpa de una fila mala, y las malas quedan registradas para revisarlas.

In [19]:
con.sql(f"""
CREATE OR REPLACE TABLE tml AS
SELECT * FROM read_csv('{D}/tml/20*.csv', all_varchar = true, store_rejects = true)
""")
sql("""
SELECT scan_id, line, error_type,
       count(*)                        AS n_errors,
       left(any_value(csv_line), 90)   AS csv_line_start
FROM reject_errors
WHERE scan_id = (SELECT max(scan_id) FROM reject_errors)  -- only the last read
GROUP BY ALL
""")

,scan_id,line,error_type,n_errors,csv_line_start
0,23,2945,MISSING COLUMNS,13,"2025-7696,Next Gen ATP Finals,Hard,8,A,I,20251..."


**Conclusión:** la **última fila de `tml/2025.csv` está cortada** (37 columnas de 50: la final de las
Next Gen ATP Finals). El fichero quedó a medio escribir en su último commit.

Además, `ATP_Database.csv` (jugadores de TML) tiene **codificación mixta**: casi todo UTF-8 pero
2 líneas en Latin-1 (p. ej. "Cinà"). Primer indicio de que TML no está bien mantenido.

In [20]:
con.sql(f"""
CREATE OR REPLACE TABLE tml_players AS
SELECT * FROM read_csv('{D}/tml/ATP_Database.csv', all_varchar = true, store_rejects = true)
""")
sql("""
SELECT scan_id, line, error_type,
       count(*)                        AS n_errors,
       left(any_value(csv_line), 90)   AS csv_line_start
FROM reject_errors
WHERE scan_id = (SELECT max(scan_id) FROM reject_errors)  -- only the last read
GROUP BY ALL
""")

,scan_id,line,error_type,n_errors,csv_line_start
0,25,2177,INVALID ENCODING,1,"""E661"",""Gonzalo Escobar"",""Gonzalo Escobar"",198..."
1,25,1372,INVALID ENCODING,1,"""C0NB"",""Federico Cina"",""Federico Cina"",2007033..."


### 8.2 ¿Hasta cuándo llega cada fuente?

In [21]:
sql("""
SELECT 'Sackmann archive'  AS source, max(tourney_date)::VARCHAR AS last_tourney_date FROM matches
UNION ALL
SELECT 'TML-Database', max(tourney_date) FROM tml
""")

,source,last_tourney_date
0,Sackmann archive,20260525
1,TML-Database,20260117


**Conclusión:** TML se **detuvo en enero de 2026** (último commit del repo: 27-ene-2026). El archivo
de Sackmann llega hasta **Roland Garros 2026**. La premisa del ADR-0002 ("TML se actualiza a
diario") **ya no es cierta**.

### 8.3 ¿Son los mismos datos? ¿Cruzan los IDs?

In [22]:
sql("""
SELECT coalesce(s.tourney_name, t.tourney_name) AS tourney, s.n AS sackmann_matches, t.n AS tml_matches
FROM (SELECT tourney_name, count(*) AS n FROM matches
      WHERE tourney_date BETWEEN 20260101 AND 20260117 GROUP BY ALL) AS s
FULL JOIN (SELECT tourney_name, count(*) AS n FROM tml
           WHERE tourney_date BETWEEN '20260101' AND '20260117' GROUP BY ALL) AS t
       USING (tourney_name)
ORDER BY tourney
""")

,tourney,sackmann_matches,tml_matches
0,Adelaide,27,27
1,Auckland,27,27
2,Brisbane,31,31
3,Hong Kong,27,27
4,United Cup,25,25


In [23]:
sql("""
(SELECT 'Sackmann archive' AS source, winner_id::VARCHAR AS sinner_id FROM matches
 WHERE winner_name = 'Jannik Sinner' LIMIT 1)
UNION ALL
(SELECT 'TML-Database', winner_id FROM tml WHERE winner_name = 'Jannik Sinner' LIMIT 1)
""")

,source,sinner_id
0,Sackmann archive,206173
1,TML-Database,S0AG


**Conclusión:** en el periodo común tienen **exactamente los mismos partidos**, pero **IDs distintos**
(Sinner: `206173` vs `S0AG`) → no se pueden mezclar sin un mapeo de jugadores.

### 8.4 ¿Qué aportan challengers y fases previas?

In [24]:
sql("""
SELECT tourney_level, count(*) AS n_matches, round(100 * avg((w_ace IS NULL)::INT), 1) AS pct_null_aces
FROM qual_chall GROUP BY ALL ORDER BY n_matches DESC
""")

,tourney_level,n_matches,pct_null_aces
0,C,15541,0.2
1,A,834,0.0
2,G,560,0.2
3,M,439,0.2


**Conclusión:** el archivo incluye ~**15.500 partidos de Challenger** (nivel `C`) en 2025–26
(5 veces más que el circuito principal) **con estadísticas** + las fases previas de los torneos
grandes. Más partidos → Elo más preciso para jugadores jóvenes que aún no juegan en el circuito
principal. Candidato para el Bloque 8.

### 8.5 Rankings semanales

In [25]:
sql("""
SELECT 'rankings_20s' AS file, count(*) AS n_rows, min(ranking_date) AS first_week,
       max(ranking_date) AS last_week, count(DISTINCT ranking_date) AS n_weeks
FROM rankings_20s
UNION ALL
SELECT 'rankings_current', count(*), min(ranking_date), max(ranking_date), count(DISTINCT ranking_date)
FROM rankings_current
""")

,file,n_rows,first_week,last_week,n_weeks
0,rankings_20s,516461,20200106,20251229,248
1,rankings_current,36468,20260105,20260608,17


**Conclusión:** el archivo trae **rankings semanales** hasta el **8-jun-2026** (TML no tiene
rankings). Útil para perfiles de jugador y validar el Elo.

## 9. Cuotas de apuestas (tennis-data.co.uk)

DuckDB lee Excel con su **extensión `excel`** (`read_xlsx`). Las extensiones amplían DuckDB
(Excel, HTTP, S3, Postgres...) y se instalan con `INSTALL` / `LOAD`.
Leemos todo como texto (`all_varchar`) para ver los datos **tal cual vienen**.

In [26]:
con.sql("INSTALL excel; LOAD excel;")
con.sql(f"""
CREATE OR REPLACE VIEW odds AS
SELECT *, 2025 AS season FROM read_xlsx('{D}/odds/2025.xlsx', all_varchar = true)
UNION ALL BY NAME
SELECT *, 2026 AS season FROM read_xlsx('{D}/odds/2026.xlsx', all_varchar = true)
""")
sql(
    "SELECT Date, Tournament, Round, Surface, Winner, Loser, B365W, B365L, PSW, PSL, AvgW, AvgL, Comment FROM odds LIMIT 5"
)

,Date,Tournament,Round,Surface,Winner,Loser,B365W,B365L,PSW,PSL,AvgW,AvgL,Comment
0,45655,Brisbane International,1st Round,Hard,Vukic A.,Goffin D.,2,1.8,2.08,1.83,2.0299999999999998,1.78,Completed
1,45656,Brisbane International,1st Round,Hard,Michelsen A.,O Connell C.,1.44,2.75,1.48,2.85,1.43,2.74,Completed
2,45656,Brisbane International,1st Round,Hard,Bonzi B.,Tabilo A.,1.67,2.2000000000000002,1.73,2.2200000000000002,1.67,2.1800000000000002,Completed
3,45656,Brisbane International,1st Round,Hard,Nishioka Y.,Rinderknech A.,1.53,2.5,1.64,2.39,1.59,2.36,Completed
4,45656,Brisbane International,1st Round,Hard,Thompson J.,Berrettini M.,2.63,1.5,2.4700000000000002,1.6,2.48,1.54,Completed


### 9.1 Trampa: ¡las fechas son números!
`Date = 45655` es un **número de serie de Excel**: días desde el 30-dic-1899. Hay que convertirlo.

In [27]:
sql("""
SELECT season, count(*) AS n_matches,
       min(DATE '1899-12-30' + TRY_CAST(Date AS INT)) AS first_date,
       max(DATE '1899-12-30' + TRY_CAST(Date AS INT)) AS last_date
FROM odds GROUP BY ALL ORDER BY season
""")

,season,n_matches,first_date,last_date
0,2025,2644,2024-12-29,2025-11-16
1,2026,2150,2026-01-04,2026-09-13


**Conclusión:** tennis-data es la **única fuente viva**: llega al **13-sep-2026**.

### 9.2 Calidad de las cuotas
`TRY_CAST` convierte a número y devuelve `NULL` si no puede (en vez de fallar). Lo necesitamos
porque algunas cuotas vienen como el texto `'-'`.

In [28]:
sql("""
SELECT season,
       round(100 * avg((TRY_CAST(B365W AS DOUBLE) IS NULL)::INT), 1) AS pct_null_bet365,
       round(100 * avg((TRY_CAST(PSW AS DOUBLE) IS NULL)::INT), 1)   AS pct_null_pinnacle,
       round(100 * avg((TRY_CAST(AvgW AS DOUBLE) IS NULL)::INT), 1)  AS pct_null_average,
       count(*) FILTER (WHERE TRY_CAST(B365W AS DOUBLE) <= 1)        AS n_invalid_bet365
FROM odds GROUP BY ALL ORDER BY season
""")

,season,pct_null_bet365,pct_null_pinnacle,pct_null_average,n_invalid_bet365
0,2025,0.5,4.2,0.0,3
1,2026,0.5,96.7,0.0,7


In [29]:
sql("SELECT Comment, count(*) AS n FROM odds GROUP BY ALL ORDER BY n DESC")

,Comment,n
0,Completed,4601
1,Retired,157
2,Walkover,36


### 9.3 El margen de la casa (*overround*)
Si una casa fuera "justa", las probabilidades implícitas (`1/cuota`) de los dos jugadores sumarían
1. Suman **más de 1**: ese exceso es el **margen** con el que gana la casa.

In [30]:
sql("""
SELECT round(100 * avg(1 / TRY_CAST(B365W AS DOUBLE) + 1 / TRY_CAST(B365L AS DOUBLE) - 1), 2) AS bet365_margin_pct,
       round(100 * avg(1 / TRY_CAST(PSW AS DOUBLE) + 1 / TRY_CAST(PSL AS DOUBLE) - 1), 2)     AS pinnacle_margin_pct,
       round(100 * avg((TRY_CAST(B365W AS DOUBLE) < TRY_CAST(B365L AS DOUBLE))::INT), 1)       AS favourite_won_pct
FROM odds
WHERE TRY_CAST(B365W AS DOUBLE) > 1 AND TRY_CAST(B365L AS DOUBLE) > 1
  AND TRY_CAST(PSW AS DOUBLE) > 1 AND TRY_CAST(PSL AS DOUBLE) > 1
""")

,bet365_margin_pct,pinnacle_margin_pct,favourite_won_pct
0,5.27,2.7,65.5


**Conclusiones:**
- Margen: Bet365 ~**5,3 %**, Pinnacle ~**2,7 %** (la casa más "afinada" del mercado).
- **El favorito de las casas gana el 65,5 %** de los partidos → **esa es la vara de medir** de
  nuestro modelo del Bloque 12.
- ⚠️ **Pinnacle desaparece en 2026** (97 % nulos) → usaremos la **media de casas** (`Avg`) como
  referencia principal y Pinnacle solo donde exista.
- Cuotas inválidas (`'-'`, `<= 1`) → nulas en silver.

### 9.4 Primer intento de *entity resolution*
tennis-data no tiene IDs: los jugadores vienen como `"Nadal R."`. Para unir con el archivo
construimos una **clave normalizada** `apellido_inicial` en ambos lados:
- minúsculas, sin acentos (`strip_accents`), solo letras → `"O Connell C."` → `oconnell_c`.
- Una **macro** (`CREATE MACRO`) es una función SQL reutilizable.

In [31]:
con.sql(
    "CREATE OR REPLACE MACRO norm(s) AS regexp_replace(lower(strip_accents(s)), '[^a-z]', '', 'g')"
)

con.sql("""
CREATE OR REPLACE VIEW match_keys AS
SELECT m.tourney_name, m.winner_name, m.loser_name,
       norm(pw.name_last) || '_' || left(norm(pw.name_first), 1) AS winner_key,
       norm(pl.name_last) || '_' || left(norm(pl.name_first), 1) AS loser_key,
       strptime(m.tourney_date::VARCHAR, '%Y%m%d')::DATE        AS tourney_start
FROM matches AS m
JOIN players AS pw ON pw.player_id = m.winner_id
JOIN players AS pl ON pl.player_id = m.loser_id
WHERE m.tourney_date >= 20241201
""")

con.sql(r"""
CREATE OR REPLACE VIEW odds_keys AS
SELECT *,
       norm(regexp_extract(Winner, '^(.*) \S+$', 1)) || '_' || left(norm(regexp_extract(Winner, ' (\S+)$', 1)), 1) AS winner_key,
       norm(regexp_extract(Loser,  '^(.*) \S+$', 1)) || '_' || left(norm(regexp_extract(Loser,  ' (\S+)$', 1)), 1) AS loser_key,
       DATE '1899-12-30' + TRY_CAST(Date AS INT) AS match_date
FROM odds
""")
sql("SELECT Winner, winner_key, Loser, loser_key FROM odds_keys LIMIT 5")

,Winner,winner_key,Loser,loser_key
0,Vukic A.,vukic_a,Goffin D.,goffin_d
1,Michelsen A.,michelsen_a,O Connell C.,oconnell_c
2,Bonzi B.,bonzi_b,Tabilo A.,tabilo_a
3,Nishioka Y.,nishioka_y,Rinderknech A.,rinderknech_a
4,Thompson J.,thompson_j,Berrettini M.,berrettini_m


Unimos por **ambas claves** + una **ventana de fechas** (el partido de cuotas debe caer entre unos
días antes y 16 días después del inicio del torneo). Solo miramos partidos anteriores al fin del
archivo (Roland Garros 2026).

In [32]:
sql("""
SELECT count(*)                                  AS odds_matches,
       count(m.winner_key)                       AS joined,
       round(100 * count(m.winner_key) / count(*), 1) AS join_rate_pct
FROM odds_keys AS o
LEFT JOIN match_keys AS m
       ON o.winner_key = m.winner_key AND o.loser_key = m.loser_key
      AND o.match_date BETWEEN m.tourney_start - 7 AND m.tourney_start + 16
WHERE o.match_date < DATE '2026-05-24'
""")

,odds_matches,joined,join_rate_pct
0,3879,3750,96.7


In [33]:
sql("""
SELECT o.match_date, o.Tournament, o.Winner, o.winner_key, o.Loser, o.loser_key
FROM odds_keys AS o
LEFT JOIN match_keys AS m
       ON o.winner_key = m.winner_key AND o.loser_key = m.loser_key
      AND o.match_date BETWEEN m.tourney_start - 7 AND m.tourney_start + 16
WHERE m.winner_key IS NULL AND o.match_date < DATE '2026-05-24'
LIMIT 10
""")

,match_date,Tournament,Winner,winner_key,Loser,loser_key
0,2025-01-02,Brisbane International,Mpetshi G.,mpetshi_g,Tiafoe F.,tiafoe_f
1,2025-04-14,Barcelona Open,Medjedovic H.,medjedovic_h,Mpetshi G.,mpetshi_g
2,2025-07-27,Canadian Open,Mpetshi G.,mpetshi_g,Mochizuki S.,mochizuki_s
3,2025-07-28,Canadian Open,Bu Y.,bu_y,Kopriva V.,kopriva_v
4,2025-08-19,Winston-Salem Open at Wake Forest University,Mpetshi G.,mpetshi_g,Martinez P.,martinez_p
5,2026-02-12,Argentina Open,Darderi L.,darderi_l,Barrios M.,barrios_m
6,2025-01-12,Australian Open,Habib H.,habib_h,Bu Y.,bu_y
7,2025-03-08,BNP Paribas Open,Medvedev D.,medvedev_d,Bu Y.,bu_y
8,2025-03-22,Miami Open,Thompson J.,thompson_j,Mpetshi G.,mpetshi_g
9,2025-04-06,Monte Carlo Masters,Thompson J.,thompson_j,Mpetshi G.,mpetshi_g


**Conclusiones:**
- Con una regla sencilla ya cruzamos el **96,7 %** (objetivo del Bloque 10: ≥ 95 % ✅).
- Los fallos tienen patrones claros que resolveremos en el Bloque 10:
  - **Apellidos compuestos**: `Mpetshi G.` ↔ Giovanni *Mpetshi Perricard*; `Alvarez Valdes L.C.`.
  - **Orden de nombres asiático**: `Bu Y.` ↔ *Yunchaokete Bu*.
  - Diferencias de transliteración.

### 9.5 ¿Cuántos partidos existen solo en tennis-data?

In [34]:
sql("""
SELECT count(*) AS matches_after_archive_end, count(DISTINCT Tournament) AS n_tournaments,
       min(match_date) AS first_date, max(match_date) AS last_date
FROM odds_keys
WHERE match_date >= DATE '2026-05-24'
""")

,matches_after_archive_end,n_tournaments,first_date,last_date
0,981,19,2026-05-24,2026-09-13


**Conclusión:** **~980 partidos** (19 torneos, de Roland Garros 2026 al 13-sep-2026, incluidos
Wimbledon y el US Open) solo existen en tennis-data. Sin estadísticas de saque, pero con ganador,
perdedor, marcador, superficie y cuotas: **suficiente para mantener el Elo al día**.

## 10. Comprobaciones de coherencia con el tenis real ✅

Si los datos reproducen hechos conocidos, podemos confiar en ellos. Si no, algo está mal
(duplicados, partidos que faltan, nombres cambiados...).

In [35]:
sql("""
SELECT 'H2H Djokovic-Nadal (expected 60 matches, 31-29)' AS check_name,
       count(*) || ' matches, ' || sum((winner_name = 'Novak Djokovic')::INT) || '-'
                || sum((winner_name = 'Rafael Nadal')::INT) AS result
FROM matches
WHERE (winner_name, loser_name) IN (('Novak Djokovic', 'Rafael Nadal'), ('Rafael Nadal', 'Novak Djokovic'))
UNION ALL
SELECT 'Djokovic Grand Slam titles (expected 24)', count(*)::VARCHAR
FROM matches WHERE tourney_level = 'G' AND round = 'F' AND winner_name = 'Novak Djokovic'
""")

,check_name,result
0,"H2H Djokovic-Nadal (expected 60 matches, 31-29)","60 matches, 31-29"
1,Djokovic Grand Slam titles (expected 24),24


### Racha de Nadal en tierra: un problema de *gaps and islands*
Queremos la racha más larga de victorias seguidas. Truco clásico (pregunta típica de entrevista):
1. Ordenamos sus partidos cronológicamente (ronda incluida: ¡lección de la sección 3!).
2. `sum(derrota) OVER (ORDER BY ...)` → un contador que **sube en cada derrota**. Todas las
   victorias entre dos derrotas comparten el mismo valor = la misma "isla".
3. Agrupamos por ese valor y contamos victorias.

In [36]:
sql("""
WITH nadal_clay AS (
    SELECT tourney_date, match_num,
           winner_name = 'Rafael Nadal' AS won,
           CASE round WHEN 'R128' THEN 1 WHEN 'R64' THEN 2 WHEN 'R32' THEN 3 WHEN 'R16' THEN 4
                      WHEN 'QF' THEN 5 WHEN 'SF' THEN 6 WHEN 'F' THEN 7 ELSE 0 END AS round_order
    FROM matches
    WHERE surface = 'Clay' AND 'Rafael Nadal' IN (winner_name, loser_name)
      AND score NOT ILIKE '%W/O%'          -- walkovers were not played
),
islands AS (
    SELECT *, sum((NOT won)::INT) OVER (ORDER BY tourney_date, round_order, match_num) AS island_id
    FROM nadal_clay
)
SELECT island_id, count(*) FILTER (WHERE won) AS winning_streak, min(tourney_date) AS streak_start
FROM islands
GROUP BY ALL
ORDER BY winning_streak DESC
LIMIT 3
""")

,island_id,winning_streak,streak_start
0,12.0,81,20050404
1,16.0,37,20090525
2,14.0,33,20080505


**Conclusión:** ✅ H2H 60 partidos 31–29, ✅ 24 Grand Slams de Djokovic, ✅ racha de **81** victorias
de Nadal en tierra (abril 2005 – mayo 2007). **Los datos del archivo son fiables.**

⚠️ Una trampa más encontrada al comprobar: el **US Open** aparece como `US Open` (1968–2019) y
`Us Open` (2020–2025) → **nunca identificar torneos por nombre**; usar el código de `tourney_id`
(`YYYY-560` = US Open).

In [37]:
sql("""
SELECT tourney_name, split_part(tourney_id, '-', 2) AS tourney_code,
       min(tourney_date // 10000) AS first_year, max(tourney_date // 10000) AS last_year
FROM matches WHERE tourney_level = 'G' GROUP BY ALL ORDER BY tourney_code, first_year
""")

,tourney_name,tourney_code,first_year,last_year
0,Roland Garros,520,1968,2026
1,Wimbledon,540,1968,2025
2,US Open,560,1968,2019
3,Us Open,560,2020,2025
4,Australian Chps.,580,1968,1968
5,Australian Open,580,1969,2026
6,Australian Open-2,581,1977,1977


## 11. Conclusiones

### Decisión de fuentes (→ ADR-0004)
| Fuente | Rol |
|---|---|
| **Archivo Sackmann** | ✅ **Principal histórica** (1968 → RG 2026): partidos, challengers, jugadores, rankings. IDs coherentes y datos verificados |
| **tennis-data.co.uk** | ✅ **Fuente viva incremental**: cuotas (evaluación del modelo) + resultados posteriores a junio 2026 para mantener el Elo al día |
| **TML-Database** | ❌ **Descartada**: sin actualizar desde enero 2026, fichero truncado, codificación mixta e IDs incompatibles |

### Trampas encontradas → dónde se resuelven
| # | Trampa | Regla / decisión | Bloque |
|---|---|---|---|
| 1 | `tourney_date` = lunes de inicio, no día del partido | `round_order`, `match_seq`, `est_match_date` | 6, 11 |
| 2 | `match_num` no es cronológico en 82 torneos | Ordenar por ronda antes que por `match_num` | 6 |
| 3 | El ganador siempre en `winner_*` | Simetrizar A/B para el modelo | 11 |
| 4 | Marcadores `RET`, `W/O`, `DEF`, `UNK`, `?` | Parser tolerante + flags; W/O no cuenta para Elo | 6, 8 |
| 5 | Sin estadísticas antes de 1991 | Elo desde 1968; modelo desde ~2000 | 8, 12 |
| 6 | Superficie nula (Davis) y `Carpet` | Solo Elo general para superficie nula | 8 |
| 7 | Valores sucios (`Alt`/`ALT`) | Normalizar en silver | 6 |
| 8 | Nombres de torneo inconsistentes (`US Open`/`Us Open`) | Identificar por código de `tourney_id` | 6 |
| 9 | Cuotas: fechas serie Excel, `'-'`, cuotas ≤ 1 | Conversión de fecha, `TRY_CAST`, nulos | 3, 6 |
| 10 | Pinnacle desaparece en 2026 | Referencia principal = media de casas | 12 |
| 11 | Sin IDs en cuotas (`"Nadal R."`) | Entity resolution (96,7 % ya con regla simple) | 10 |
| 12 | Ficheros rotos en origen (TML) | Cuarentena de filas inválidas (`store_rejects`) | 3, 5 |